# Deep Dive into LangChain
## Tools and Agents + ReAct

This notebook uses the **langchain-classic** library.

Make sure to import from **langchain_classic** rather than **langchain**.

In [ ]:
pip install -r ./requirements.txt -q

Note: you may need to restart the kernel to use updated packages.


Download [requirements.txt](https://drive.google.com/file/d/1oek3_JLI97qy4YZDc2UmmaKXfH7n9FJw/view?usp=sharing)

In [2]:
import os
from dotenv import load_dotenv, find_dotenv

# loading the API Keys from .env
load_dotenv(find_dotenv(), override=True)

# os.environ.get('OPENAI_API_KEY')

True

In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI()  

# invoking the llm (running the prompt)
output = llm.invoke('Explain quantum mechanics in one sentence.', model='gpt-4o-mini', temperature=0.1)
print(output.content)

Quantum mechanics is a fundamental theory in physics that describes the behavior of matter and energy at the smallest scales, where particles exhibit wave-particle duality, uncertainty, and entanglement.


In [6]:
# help(ChatOpenAI)  # see the llm constructor arguments with its defaults

In [4]:
# using Chat Completions API Messages: System, Assistant and Human

from langchain_classic.schema import(
    SystemMessage, 
    AIMessage,
    HumanMessage
)
messages = [
    SystemMessage(content='You are a physicist and respond only in German.'),
    HumanMessage(content='Explain quantum mechanics in one sentence.')
]

output = llm.invoke(messages)
print(output.content)

Quantenmechanik beschreibt das Verhalten von Teilchen auf der kleinsten Skala und beruht auf quantisierten physikalischen Größen wie Energie und Impuls.


In [5]:
%%time
prompt = 'Tell a me a joke that a toddler can understand.'
llm.invoke(prompt)

CPU times: user 5.5 ms, sys: 55 μs, total: 5.56 ms
Wall time: 955 ms


AIMessage(content='Why did the teddy bear say no to dessert? Because he was stuffed!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 18, 'total_tokens': 34, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DJyqEO61aMQ6VTfE3BCAM1FiTtK56', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cf60b-8bda-75c1-b307-7e9e823a8f11-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 16, 'total_tokens': 34, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [6]:
%%time
llm.invoke(prompt)

CPU times: user 4.19 ms, sys: 1.01 ms, total: 5.2 ms
Wall time: 1.78 s


AIMessage(content='Why did the teddy bear say no to dessert? Because she was already stuffed!', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 18, 'total_tokens': 35, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DJyqGSfXXqCrRMJFxQ6hbfC4nD0PV', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cf60b-95fe-75a1-9992-0dfa7dc68021-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 18, 'output_tokens': 17, 'total_tokens': 35, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## LangChain Agents in Action: Python REPL

In [ ]:
# pip install -q --upgrade langchain_experimental pydantic

Note: you may need to restart the kernel to use updated packages.


In [7]:
from langchain_experimental.utilities import PythonREPL
python_repl = PythonREPL()
python_repl.run('print([n for n in range(1, 100) if n % 13 == 0])')

Python REPL can execute arbitrary code. Use with caution.


'[13, 26, 39, 52, 65, 78, 91]\n'

In [8]:
# Updated code 

from langchain_openai import ChatOpenAI
from langchain_experimental.tools.python.tool import PythonREPLTool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent

llm = ChatOpenAI(model="gpt-5-nano", temperature=0)
tools = [PythonREPLTool()]

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a Python coder. Use the Python REPL for any math/code. Return only the final answer."),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

agent_executor.invoke({
    "input": "Calculate the square root of the factorial of 12 and display it with 4 decimal points"
})




> Entering new AgentExecutor chain...

Invoking: `Python_REPL` with `{'query': "import math; val = math.factorial(12); res = math.sqrt(val); print(val, res, f'{res:.4f}')"}`


479001600 21886.105181141756 21886.1052
21886.1052

> Finished chain.


{'input': 'Calculate the square root of the factorial of 12 and display it with 4 decimal points',
 'output': '21886.1052'}

In [9]:
response = agent_executor.invoke({
    'input': 'What is the answer to 5.1 ** 7.3?'
})



> Entering new AgentExecutor chain...

Invoking: `Python_REPL` with `{'query': 'print(5.1 * 7.3)'}`


37.23

Invoking: `Python_REPL` with `{'query': 'print(5.1 ** 7.3)'}`


146306.05007233328
146306.05007233328

> Finished chain.


In [10]:
response

{'input': 'What is the answer to 5.1 ** 7.3?', 'output': '146306.05007233328'}

In [11]:
print(response['input'])

What is the answer to 5.1 ** 7.3?


In [12]:
print(response['output'])


146306.05007233328


## LangChain Tools: DuckDuckGo and Wikipedia

### DuckDuckGo Search

In [13]:
pip install -q duckduckgo-search

Note: you may need to restart the kernel to use updated packages.


In [14]:
pip install -q ddgs

Note: you may need to restart the kernel to use updated packages.


In [15]:
# optional, filter workings (recommended in production)
import warnings
warnings.filterwarnings('ignore', module='langchain')

In [17]:
from langchain_classic.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()
output = search.invoke('Where was Freddie Mercury born?')
print(output)

FreddieMercury(bornFarrokh Bulsara; 5 September 1946 – 24 November 1991)wasa British singer and songwriter who achieved global fame as the lead vocalist and pianist of the rock band Queen. FreddieMercurywasbornon the Tanzanian island of Zanzibar. His parents, Bomi and Jer Bulsara, sent him off to a private school in India, from 1955 til 1963. In 1964, he and his family flew to England. In 1966 he started his education at the Ealing College of Art,wherehe graduated in 1969. Whereand whenwasFreddieMercuryborn?FreddieMercurywasbornFarrokh Bulsara in Stone Town in the British protectorate of the Sultanate of Zanzibar, East Africa (now part of Tanzania) on September 5, 1946. WhatwasFreddieMercury’s education?BornFarrokh Bulsara in 1946 to Parsi parents in Zanzibar ,FreddieMercuryattended boarding school in Panchgani, Maharashtra , India ,wherehe took piano lessons. WherewasFreddieMercurybornand whatwashis birth name? The man known to millions asFreddieMercurywasbornFarrokh Bulsara on 5th Se

In [18]:
search.name

'duckduckgo_search'

In [19]:
search.description

'A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.'

In [20]:
from langchain_classic.tools import DuckDuckGoSearchResults
search = DuckDuckGoSearchResults()
output = search.run('Freddie Mercury and Queen.')
print(output)

snippet: 1 week ago -Freddie Mercury (born Farrokh Bulsara; 5 September 1946 – 24 November 1991) was a British singer and songwriter who achieved global fame as thelead vocalist and pianist of the rock band Queen. Regarded as one of the greatest singers in the history of rock music, he is known for his flamboyant ..., title: Freddie Mercury - Wikipedia, link: https://en.wikipedia.org/wiki/Freddie_Mercury, snippet: 5 days ago -Bohemian Rhapsodyis a 2018 biographical drama film that focuses on the life of Freddie Mercury, the lead singer of the British rock band Queen, from the formation of the band in 1970 to their 1985 Live Aid performance at the original Wembley Stadium. It was directed by Bryan Singer from a ..., title: Bohemian Rhapsody (film) - Wikipedia, link: https://en.wikipedia.org/wiki/Bohemian_Rhapsody_(film), snippet: 1 day ago -Brian May's website also stated that Rodgers would be "featured with" Queen as "Queen + Paul Rodgers", not replacing Mercury. In November 2004, Quee

In [21]:
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

wrapper = DuckDuckGoSearchAPIWrapper(region='de-de', max_results=3, safesearch='moderate')
search = DuckDuckGoSearchResults(api_wrapper=wrapper, source='news')
output = search.run('Berlin')

In [22]:
print(output)

snippet: Berlin ([bɛr'li:n] ) ist die Hauptstadt und ein Land der Bundesrepublik Deutschland. Die Großstadt ist mit rund 3,7 Millionen Einwohnern die bevölkerungsreichste und mit 891 Quadratkilometern die flächengrößte Gemeinde Deutschlands sowie die bevölkerungsreichste Stadt der Europäischen Union. In der Agglomeration Berlin leben mehr als 4,7 Millionen Menschen und in der Metropolregion Berlin-Brandenburg mehr als 6,2 Millionen. Der Stadtstaat besteht aus zwölf Bezirken. Neben den Flüssen Spree, Havel und Dahme befinden sich im Stadtgebiet kleinere Fließgewässer sowie zahlreiche Seen und Wälder.Berlin wurde im 13. Jahrhundert erstmals urkundlich erwähnt. Die Stadt war Hauptstadt der Mark Brandenburg, Preußens und Deutschlands. Im Laufe des 19. und frühen 20. Jahrhunderts entwickelte sich Berlin zur weltweit drittgrößten Stadt. Nach dem Ende des Zweiten Weltkriegs unterlag die Stadt 1945 dem Viermächtestatus; Ost-Berlin war ab 1949 de facto Hauptstadt der Deutschen Demokratischen Re

In [24]:
import re

pattern = r"snippet:\s*(.*?)\s*,\s*title:\s*(.*?)\s*,\s*link:\s*(https?://[^\s,]+)"

matches = re.findall(pattern, output, re.DOTALL)

for snippet, title, link in matches:
    print(f"Snippet: {snippet}\nTitle: {title}\nLink: {link}\n")
    print("-" * 50)

Snippet: Berlin ([bɛr'li:n] ) ist die Hauptstadt und ein Land der Bundesrepublik Deutschland. Die Großstadt ist mit rund 3,7 Millionen Einwohnern die bevölkerungsreichste und mit 891 Quadratkilometern die flächengrößte Gemeinde Deutschlands sowie die bevölkerungsreichste Stadt der Europäischen Union. In der Agglomeration Berlin leben mehr als 4,7 Millionen Menschen und in der Metropolregion Berlin-Brandenburg mehr als 6,2 Millionen. Der Stadtstaat besteht aus zwölf Bezirken. Neben den Flüssen Spree, Havel und Dahme befinden sich im Stadtgebiet kleinere Fließgewässer sowie zahlreiche Seen und Wälder.Berlin wurde im 13. Jahrhundert erstmals urkundlich erwähnt. Die Stadt war Hauptstadt der Mark Brandenburg, Preußens und Deutschlands. Im Laufe des 19. und frühen 20. Jahrhunderts entwickelte sich Berlin zur weltweit drittgrößten Stadt. Nach dem Ende des Zweiten Weltkriegs unterlag die Stadt 1945 dem Viermächtestatus; Ost-Berlin war ab 1949 de facto Hauptstadt der Deutschen Demokratischen Re

### Wikipedia

In [ ]:
# pip install -q wikipedia

Note: you may need to restart the kernel to use updated packages.


In [25]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [28]:
api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=10000)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper)
wiki.invoke({'query': 'llamaindex'})

'Page: Vector database\nSummary: A vector database, vector store or vector search engine is a database that stores and retrieves embeddings of data in vector space. Vector databases typically implement approximate nearest neighbor algorithms so users can search for records semantically similar to a given input, unlike traditional databases which primarily look up records by exact match. Use-cases for vector databases include similarity search, semantic search, multi-modal search, recommendations engines, object detection, and retrieval-augmented generation (RAG).\nVector embeddings are mathematical representations of data in a high-dimensional space. In this space, each dimension corresponds to a feature of the data, with the number of dimensions ranging from a few hundred to tens of thousands, depending on the complexity of the data being represented. Each data item is represented by one vector in this space. Words, phrases, or entire documents, as well as images, audio, and other typ

In [29]:
output = wiki.invoke('Google Gemini')

In [30]:
print(output)

Page: Google Gemini
Summary: Gemini (also known as Google Gemini and formerly known as Bard) is a generative artificial intelligence chatbot and virtual assistant developed by Google. It is powered by the large language model (LLM) of the same name, after previously being based on LaMDA and PaLM 2.
The Gemini architecture is trained natively on multiple data types, allowing the models to process and generate text, computer code, images, audio, and video simultaneously. Google distributes the technology in varying capacities, ranging from efficient on-device versions ("Nano") and cost-effective, high-throughput variants ("Flash") to high-compute models designed for complex reasoning ("Pro" and "Ultra"). The 1.5 and 3 model generations introduced extended context windows, enabling the analysis of large datasets such as entire codebases, long-form videos, or extensive document archives in a single prompt.
Gemini was first announced on December 6, 2023, and replaced existing Google brandin

## Creating a ReAct Agent

In [33]:
pip install langchainhub -q

Note: you may need to restart the kernel to use updated packages.


In [57]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

True

In [ ]:
from langchain_classic.prompts import PromptTemplate
from langchain_classic import hub
from langchain_classic.agents import Tool, AgentExecutor, initialize_agent, create_react_agent
from langchain_classic.tools import DuckDuckGoSearchRun, WikipediaQueryRun
from langchain_classic.utilities import WikipediaAPIWrapper
from langchain_experimental.tools.python.tool import PythonREPLTool
from langchain_openai import ChatOpenAI

# Initialize the ChatOpenAI model 
# Use a model that supports the stop parameter
llm = ChatOpenAI(model_name='gpt-4o-mini', temperature=0)

# Define a template for answering questions
template = '''
Answer the following questions in GERMAN as best you can.
Questions: {q}
'''

# Create a PromptTemplate object from the template
prompt_template = PromptTemplate.from_template(template)

# Pull the react prompt from the hub
prompt = hub.pull('hwchase17/react', 
                  api_key=os.getenv("LANGSMITH_API_KEY"))

# displaying information about the react prompt
# print(type(prompt))
# print(prompt.input_variables)
# print(prompt.template)


# Create tools for the agent

# 1. Python REPL Tool (for executing Python code)
python_repl = PythonREPLTool()
python_repl_tool = Tool(
    name='Python REPL',
    func=python_repl.run,
    description='Useful when you need to use Python to answer a question. You should input Python code.'
)

# 2. Wikipedia Tool (for searching Wikipedia)
api_wrapper = WikipediaAPIWrapper()
wikipedia = WikipediaQueryRun(api_wrapper=api_wrapper)
wikipedia_tool = Tool(
    name='Wikipedia',
    func=wikipedia.run,
    description='Useful for when you need to look up a topic, country, or person on Wikipedia.'
)

# 3. DuckDuckGo Search Tool (for general web searches)
search = DuckDuckGoSearchRun()
duckduckgo_tool = Tool(
    name='DuckDuckGo Search',
    func=search.run,
    description='Useful for when you need to perform an internet search to find information that another tool can\'t provide.'
)

# Combine the tools into a list
tools = [python_repl_tool, wikipedia_tool, duckduckgo_tool]

# Create a react agent with the ChatOpenAI model, tools, and prompt
agent = create_react_agent(llm, tools, prompt)

# Initialize the AgentExecutor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=10
)


In [77]:
# question = 'Generate the first 20 numbers in the Fibonacci series.'
# question = 'Who is the current prime minister of the UK?'
question = 'Tell me about Napoleon Bonaparte early life'
output = agent_executor.invoke({
    'input': prompt_template.format(q=question)
})




> Entering new AgentExecutor chain...
Ich sollte Informationen über das frühe Leben von Napoleon Bonaparte in deutscher Sprache bereitstellen. Um genaue und umfassende Informationen zu erhalten, werde ich Wikipedia verwenden, da es eine zuverlässige Quelle für biografische Informationen ist.

Action: Wikipedia  
Action Input: "Napoleon Bonaparte frühes Leben"  Page: Neuenwalde Convent
Summary: The Neuenwalde Convent (N. Low Saxon: Klooster Niewohl, German: Kloster Neuenwalde; Latin: Conventus Sanct[a]e Crucis) is a Lutheran damsels' convent in Neuenwalde, a locality of Geestland, Lower Saxony, Germany.
Since 1683 the convent is owned by the corporation of the Bremian Knighthood and used for Lutheran conventuals and continues to function as such today. It is the only convent preserved in the Elbe-Weser triangle out of a former sample of 14 monasteries. The convent was established as a Roman Catholic nunnery in 1219, and was recorded in 1282 for pursuing the Benedictine observance. The

In [51]:
print(output['input'])


Answer the following questions in GERMAN as best you can.
Questions: Tell me about Napoleon Bonaparte early life



In [52]:
output['output']

'Napoleon Bonaparte wurde am 15. August 1769 auf der Insel Korsika geboren. Er stammte aus einer Familie italienischer Herkunft und zog 1779 nach Festlandfrankreich. Im Jahr 1785 wurde er als Offizier in die französische Royal-Armee aufgenommen. Napoleon unterstützte die Französische Revolution, die 1789 begann, und förderte deren Anliegen auf Korsika. Er stieg schnell in den militärischen Rängen auf, nachdem er 1793 die Belagerung von Toulon gewonnen hatte. 1795 besiegte er royalistische Aufständische in Paris. Im Jahr 1796 übernahm er das Kommando über eine Militärkampagne gegen die Österreicher und deren italienische Verbündete, was ihn zu einem nationalen Helden machte.'